# USGS 3DEP Elevation (EPQS) — Terrain for a Field

**What it is.** The USGS Elevation Point Query Service returns ground elevation from the
3D Elevation Program (3DEP) for any lon/lat — no key.

| | |
|---|---|
| **Coverage** | United States (1–10 m 3DEP) |
| **Cost / key** | **Free · no key** |
| **Docs** | https://apps.nationalmap.gov/epqs/ |

**Why it's in Tracera.** Elevation → **slope & aspect**, which drive drainage, erosion risk,
frost pockets, and variable-rate zones. A lightweight terrain layer for any field.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — elevation at the field

In [2]:
EPQS = "https://epqs.nationalmap.gov/v1/json"

def elevation(lat=LAT, lon=LON):
    r = requests.get(EPQS, params={"x": lon, "y": lat, "units": "Meters",
                     "wkid": 4326, "includeDate": False}, timeout=60)
    r.raise_for_status()
    return float(r.json()["value"])

e = elevation()
print(f"Elevation at the field: {e:.1f} m")
assert e > 0

Elevation at the field: 296.5 m


### 2. Derived — approximate slope from a small elevation cross

In [3]:
import math

# Sample N/S/E/W ~150 m away and estimate slope (%) and aspect.
d = 0.0014  # ~150 m in degrees latitude
north, south = elevation(LAT + d, LON), elevation(LAT - d, LON)
east, west = elevation(LAT, LON + d), elevation(LAT, LON - d)
span_m = 2 * d * 111_000
dz_ns, dz_ew = (north - south), (east - west)
slope_pct = math.hypot(dz_ns, dz_ew) / span_m * 100
aspect = (math.degrees(math.atan2(dz_ew, dz_ns)) + 360) % 360
terrain = pd.Series({"center_m": elevation(), "north_m": north, "south_m": south,
                     "east_m": east, "west_m": west,
                     "slope_pct": round(slope_pct, 2), "aspect_deg": round(aspect, 0)})
terrain

center_m      296.492889
north_m       298.999451
south_m       295.917755
east_m        297.341675
west_m        295.771301
slope_pct       1.110000
aspect_deg     27.000000
dtype: float64

### Notes & how to extend
- **Field-wide terrain:** for real slope/aspect maps, download a 3DEP DEM tile
  (`apps.nationalmap.gov`/TNM Access API) and compute with `richdem`/`gdaldem`, rather than
  point-by-point EPQS calls.
- **Rate:** EPQS is one point per request; cache results for fixed field locations.
- Pair elevation zones with **SSURGO** drainage class to map wet/dry areas of a field.